# Introduzione al notebook
In questo notebook sperimentiamo la **quantizzazione** di modelli LLM e la loro esecuzione locale su CPU usando **Ollama** e modelli in formato **GGUF**.

Concetti teorici:
- **Quantizzazione**: riduzione della precisione numerica (es. da 16 bit a 8 o 4 bit) per diminuire memoria e tempo di calcolo.
- **Inferenza locale**: il modello gira sul PC dello studente, senza inviare dati al cloud, garantendo privacy.
- **Formato GGUF**: struttura ottimizzata per modelli compatibili con `llama.cpp` e Ollama.

Obiettivo:
- Scaricare modelli già quantizzati
- Eseguire inferenze
- Misurare prestazioni e qualità
- Riflettere sul compromesso tra accuratezza e velocità

## Installazione e verifica di Ollama
Concetto teorico:
- Ollama è un runtime per modelli LLM ottimizzato per CPU, compatibile con GGUF.
- Funziona anche su PC senza GPU, sfruttando la libreria `llama.cpp`.

Cosa fa il codice:
- Verifica che Ollama sia installato e funzionante, mostrando la versione.
- Se non installato, lo studente deve scaricarlo dal sito ufficiale.

Cosa si può capire:
- Conferma dell'ambiente di lavoro.
- Assicurarsi che il runtime sia pronto per scaricare e usare i modelli.

In [7]:
!pip install ollama

In [2]:
!ollama --version

ollama version is 0.34.4


## Download di modelli quantizzati in GGUF
Concetto teorico:
- I modelli LLM possono essere distribuiti in vari formati e livelli di quantizzazione.
- La quantizzazione riduce dimensioni e velocizza l’inferenza, ma può degradare leggermente la qualità.

Cosa fa il codice:
- Scarica due versioni dello stesso modello LLaMA 2 da 3B parametri:
  - `q8_0` (8 bit)
  - `q4_0` (4 bit)
- Queste versioni serviranno per confrontare prestazioni e qualità.

Cosa si può capire:
- Differenze di dimensione e velocità tra INT8 e INT4.
- Effetto della quantizzazione sul tempo di download e spazio disco.

In [3]:
# Scarica i modelli
!ollama pull llama3.2:3b-instruct-q8_0
!ollama pull llama3.2:3b-instruct-q4_0
!ollama pull llama3.2:3b-instruct-q2_K

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling ed5cd7dbde6e: 100% ▕██████████████████▏ 3.4 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling eb854b581f54: 100% ▕██████████████████▏  559 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 9c8a9ab5edab: 100% ▕██████████████████▏ 1.9 

In [ ]:
# !ollama pull gpt-oss:20b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 585 KB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 2.3 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 4.2 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 7.7 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 9.1 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏  13 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏  17 MB/ 13 GB        

In [4]:
# Lista i modelli scaricati
!ollama list

NAME                         ID              SIZE      MODIFIED       
llama3.2:3b-instruct-q2_K    43e1dcc10fd4    1.4 GB    6 seconds ago     
llama3.2:3b-instruct-q4_0    9b9453afbdd6    1.9 GB    7 seconds ago     
llama3.2:3b-instruct-q8_0    e410b836fe61    3.4 GB    8 seconds ago     
llama3.2:latest              a80c4f17acd5    2.0 GB    44 minutes ago    


## Prima inferenza di prova
Concetto teorico:
- L'inferenza è il processo di generazione di testo a partire da un prompt.
- Su CPU, la velocità dipende da quantizzazione, dimensione modello e ottimizzazione.

Cosa fa il codice:
- Definisce una funzione per eseguire un prompt con Ollama.
- Misura tempo di esecuzione e memoria RAM usata.
- Esegue un prompt semplice (riassunto sulla fotosintesi).

Cosa si può capire:
- Quanto velocemente un modello quantizzato risponde.
- Quanta RAM è richiesta in un caso reale.
- Relazione tra prompt, tempo di risposta e consumo di memoria.

In [11]:
import ollama 
import time
import psutil

def run_ollama_prompt(model_name, prompt):
    start_time = time.time()
    process = psutil.Process()  # processo corrente (Python)
    mem_used = 0

    # Esegui prompt con Ollama (API Python)
    response = ""
    response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}]
    )

    mem_used = max(mem_used, process.memory_info().rss / (1024**2))

    elapsed = time.time() - start_time
    return response["message"]["content"].strip(), elapsed, mem_used

# model_gpt = 'gpt-oss:20b'
model8 = "llama3.2:3b-instruct-q8_0"
model4 = "llama3.2:3b-instruct-q4_0"

models = [model8, model4]

for model in models:
    output, elapsed, mem_used = run_ollama_prompt(
        model,
        "Raccontami 3 barzellette."
    )

    print(f"\n=== RISPOSTA {model } ===")
    print(output)
    print(f"\nTempo totale: {elapsed:.2f} s")
    print(f"Memoria massima: {mem_used:.2f} MB")


=== RISPOSTA llama3.2:3b-instruct-q8_0 ===
Ecco tre barzellette per te:

1. **Perché la gallina non vuole andare al bar?**

Perché vuole bere solo acqua!

2. **Perché il computer è andato al dottore?**

Perché aveva un virus!

3. **Perché il pisello non vuole sposare?**

Perché è ancora troppo verde!

Spero che ti abbiano fatto sorridere!

Tempo totale: 20.60 s
Memoria massima: 73.71 MB

=== RISPOSTA llama3.2:3b-instruct-q4_0 ===
Certo, ecco tre barzellette per te!

**Barzelletta 1**

Un uomo entra in un bar e ordinò un bicchiere di vino. Mentre beveva, notò che il barista stava guardandolo con un'espressione strana.

"Perché mi stai guardando così?" chiese l'uomo.

"Non ti sto guardando, sono in piedi dietro il bancone e devo scrivere il tuo nome sul biglietto della presa!"

**Barzelletta 2**

Un amico chiede a un altro: "Cosa ti piace fare nel tempo libero?"

"Le giostre, le fiere, le mostre... e il cinema, naturalmente!"

"Ah, ma che cosa ti piace vedere al cinema?"

"Le storie di 

## Confronto tra modelli INT8 e INT4
Concetto teorico:
- I modelli quantizzati a 8 bit mantengono più precisione, ma sono più lenti e pesanti.
- I modelli a 4 bit sono più leggeri e veloci, ma possono perdere qualità.

Cosa fa il codice:
- Esegue una serie di prompt su entrambi i modelli.
- Registra tempo, memoria e risposta.
- Organizza i risultati in una tabella.

Cosa si può capire:
- Differenze di latenza e RAM tra 8 bit e 4 bit.
- Variazioni qualitative nelle risposte.
- Osservare il compromesso tra prestazioni e accuratezza.

In [12]:
prompts = [
    "Spiega cos'è la quantizzazione nei modelli di intelligenza artificiale.",
    "Traduci in inglese: La scuola è finita.",
    "Quali sono i principali vantaggi dell'inferenza locale?"
]

results = []

for precision_model in [model8, model4]:
    for p in prompts:
        out, t, mem = run_ollama_prompt(precision_model, p)
        results.append({
            "modello": precision_model,
            "prompt": p,
            "tempo_s": t,
            "memoria_MB": mem,
            "risposta": out
        })

import pandas as pd
df = pd.DataFrame(results)
df

,modello,prompt,tempo_s,memoria_MB,risposta
0,llama3.2:3b-instruct-q8_0,Spiega cos'è la quantizzazione nei modelli di ...,45.500268,73.738281,La quantizzazione è un concetto fondamentale n...
1,llama3.2:3b-instruct-q8_0,Traduci in inglese: La scuola è finita.,2.949651,73.746094,"La traduzione in inglese è:\n\n""The school is ..."
2,llama3.2:3b-instruct-q8_0,Quali sono i principali vantaggi dell'inferenz...,23.013763,73.753906,Gli inferitori locali sono dispositivi di assi...
3,llama3.2:3b-instruct-q4_0,Spiega cos'è la quantizzazione nei modelli di ...,26.262498,73.765625,La quantizzazione è un concetto fondamentale n...
4,llama3.2:3b-instruct-q4_0,Traduci in inglese: La scuola è finita.,0.457291,73.785156,"La traduzione in inglese è: ""School is out."""
5,llama3.2:3b-instruct-q4_0,Quali sono i principali vantaggi dell'inferenz...,10.182427,73.785156,L'inferenza locale è un concetto matematico ch...


# Laboratorio: Quantizzazione dei modelli LLM (4-bit vs 8-bit)

## 🧩 Esercizio 2 – Misurare la velocità di inferenza

**Obiettivo:** comprendere come la quantizzazione influisce sulla **velocità di risposta** del modello.

**Istruzioni:**
1. Usa lo **stesso prompt breve**, ad esempio:
   > "Chi ha scritto *I Promessi Sposi*?"
2. Misura il **tempo totale di generazione della risposta** per ciascuna variante: **modello 8-bit** e **modello 4-bit**.
3. Ripeti la misura **3 volte** per ogni modello e calcola la **media**.

💡 *Suggerimento tecnico:* puoi misurare il tempo con:
```python
import time

prompt = "Chi ha scritto I Promessi Sposi?"

# Misura per il modello 8-bit
t8 = []
for _ in range(3):
    start = time.time()
    _ = model_8bit(prompt)
    t8.append(time.time() - start)
print(f"Tempo medio 8-bit: {sum(t8)/3:.2f} s")

# Misura per il modello 4-bit
t4 = []
for _ in range(3):
    start = time.time()
    _ = model_4bit(prompt)
    t4.append(time.time() - start)
print(f"Tempo medio 4-bit: {sum(t4)/3:.2f} s")
```

📘 *Domande guida:*
- Quale versione è più veloce in media?
- Il guadagno di velocità corrisponde a  “riduzione dei tempi di inferenza”?

In [17]:
import time
import numpy as np
import ollama

# Configurazione del prompt e del numero di ripetizioni
prompt = "Chi ha scritto I Promessi Sposi?"
num_runs = 3

# Modelli da confrontare (scaricati nel notebook)
models = {
    "Modello 8-bit": "llama3.2:3b-instruct-q8_0",
    "Modello 4-bit": "llama3.2:3b-instruct-q4_0",
}

results = {}

print(f"--- INIZIO TEST DI INFERENZA ---")
print(f"Prompt utilizzato: '{prompt}'\n")

for label, model_name in models.items():
    print(f"--- Misurazione per {label} ({model_name}) ---")
    times = []

    for i in range(num_runs):
        start_time = time.time()

        # Generazione della risposta tramite Ollama
        response = ollama.generate(model=model_name, prompt=prompt)

        end_time = time.time()
        elapsed_time = end_time - start_time
        times.append(elapsed_time)

        print(f"  Esecuzione {i+1}: {elapsed_time:.3f} secondi")

    avg_time = np.mean(times)
    results[label] = {"times": times, "avg": avg_time}
    print(f"  --> TEMPO MEDIO ({label}): {avg_time:.3f} secondi\n")

# Tabella riassuntiva dei risultati
print("=" * 45)
print(f"{'Modello':<18} | {'Tempi (s)':<15} | {'Media (s)':<8}")
print("-" * 45)
for label, data in results.items():
    times_str = ", ".join([f"{t:.2f}" for t in data["times"]])
    print(f"{label:<18} | {times_str:<15} | {data['avg']:.3f}")
print("=" * 45)

--- INIZIO TEST DI INFERENZA ---
Prompt utilizzato: 'Chi ha scritto I Promessi Sposi?'

--- Misurazione per Modello 8-bit (llama3.2:3b-instruct-q8_0) ---
  Esecuzione 1: 9.461 secondi
  Esecuzione 2: 0.964 secondi
  Esecuzione 3: 13.489 secondi
  --> TEMPO MEDIO (Modello 8-bit): 7.971 secondi

--- Misurazione per Modello 4-bit (llama3.2:3b-instruct-q4_0) ---
  Esecuzione 1: 10.499 secondi
  Esecuzione 2: 0.867 secondi
  Esecuzione 3: 0.495 secondi
  --> TEMPO MEDIO (Modello 4-bit): 3.954 secondi

Modello            | Tempi (s)       | Media (s)
---------------------------------------------
Modello 8-bit      | 9.46, 0.96, 13.49 | 7.971
Modello 4-bit      | 10.50, 0.87, 0.49 | 3.954


## 🧩 Esercizio 3 – Valutazione qualitativa della risposta (accuratezza)

**Obiettivo:** osservare l’effetto della quantizzazione sulla **qualità linguistica** delle risposte.

**Istruzioni:**
Prepara **5 prompt**, uno per ciascun tipo:
1. Domanda fattuale breve  
2. Spiegazione di un concetto semplice  
3. Riassunto di un testo di 6–8 frasi  
4. Produzione creativa (es. inizio di un racconto)  
5. Ragionamento leggero (es. “Se Marco ha 3 mele e ne dà una a Luca…”)

Esegui i prompt su **modello 8-bit** e **modello 4-bit**, confrontando:
- coerenza del testo,  
- completezza delle risposte,  
- correttezza delle informazioni.

In [18]:
import ollama

# 1. Definizione dei 5 prompt richiesti dall'esercizio
prompts = [
    # 1. Domanda fattuale breve
    "Chi ha inventato il telefono?",
    # 2. Spiegazione di un concetto semplice
    "Spiega in modo semplice cos'è una rete neurale.",
    # 3. Riassunto di un testo di 6-8 frasi
    (
        "Riassumi il seguente testo in una sola frase: "
        "L'esplorazione dello spazio ha da sempre affascinato l'umanità. Fin"
        " dai tempi antichi, le persone hanno guardato le stelle con"
        " meraviglia. Con l'avvento della tecnologia moderna, siamo stati in"
        " grado di inviare sonde su pianeti lontani. La Luna è stata la prima"
        " meta raggiunta dall'uomo nel 1969. Oggi l'attenzione si è spostata"
        " su Marte e sulle sue risorse potenziali. Le agenzie spaziali di"
        " tutto il mondo collaborano per costruire basi permanenti."
        " Il futuro del viaggio spaziale dipenderà dalle nuove tecnologie di"
        " propulsione."
    ),
    # 4. Produzione creativa
    "Scrivi l'inizio di una storia ambientata su Marte.",
    # 5. Ragionamento leggero
    "Se Marco ha 3 mele e ne dà una a Luca, quante mele gli rimangono?",
]

models = {"8bit": "llama3.2:3b-instruct-q8_0", "4bit": "llama3.2:3b-instruct-q4_0"}

risposte = {"8bit": [], "4bit": []}

# 2. Esecuzione dei prompt
print("Esecuzione dei prompt in corso...\n")
for key, model_name in models.items():
  print(f"--- Generazione risposte per il modello {key} ({model_name}) ---")
  for i, p in enumerate(prompts, 1):
    res = ollama.generate(model=model_name, prompt=p)
    risposte[key].append(res["response"].strip())
    print(f"Prompt {i} completato.")
  print()

# 3. Visualizzazione affiancata dei risultati
print("=" * 80)
print("CONFRONTO RISPOSTE: MODELLO 8-BIT vs MODELLO 4-BIT")
print("=" * 80)

for idx, p in enumerate(prompts, 1):
  print(f"\n--- PROMPT {idx} ---")
  print(f"Testo: {p[:70]}..." if len(p) > 70 else f"Testo: {p}")
  print("-" * 80)
  print(f"[8-BIT]:\n{risposte['8bit'][idx-1]}\n")
  print(f"[4-BIT]:\n{risposte['4bit'][idx-1]}")
  print("=" * 80)

Esecuzione dei prompt in corso...

--- Generazione risposte per il modello 8bit (llama3.2:3b-instruct-q8_0) ---
Prompt 1 completato.
Prompt 2 completato.
Prompt 3 completato.
Prompt 4 completato.
Prompt 5 completato.

--- Generazione risposte per il modello 4bit (llama3.2:3b-instruct-q4_0) ---
Prompt 1 completato.
Prompt 2 completato.
Prompt 3 completato.
Prompt 4 completato.
Prompt 5 completato.

CONFRONTO RISPOSTE: MODELLO 8-BIT vs MODELLO 4-BIT

--- PROMPT 1 ---
Testo: Chi ha inventato il telefono?
--------------------------------------------------------------------------------
[8-BIT]:
Ciao! Sono felice di aiutarti a scoprire chi ha inventato il telefono.

Il telefono è stato inventato da Alexander Graham Bell, un ingegnere e scienziato scozzese-Canadese, nel 1876. Bell lavorava alla National Academy of Sciences a Washington, D.C., negli Stati Uniti, all'epoca.

Secondo la leggenda, Bell stava cercando di creare un dispositivo che potesse trasmettere il suono in tempo reale, in mod

## 🧩 Esercizio 4 – Prompt di stress (robustezza con input lunghi)

**Obiettivo:** valutare come la **quantizzazione più spinta (4-bit)** reagisce a input complessi.

**Istruzioni:**
1. Prendi un testo di circa **200–300 parole** (es. un breve articolo o un estratto da Wikipedia).
2. Crea un prompt che chieda di **riassumere** o **estrarre i punti chiave**.
3. Esegui lo stesso prompt con **modello 8-bit** e **modello 4-bit**.

💡 *Suggerimento tecnico:*
```python
long_prompt = "Riassumi il seguente testo in 5 punti chiave:\n\n[TUO TESTO QUI]"
print("Risposta 8-bit:")
print(model_8bit(long_prompt))
print("\nRisposta 4-bit:")
print(model_4bit(long_prompt))
```

📘 *Domande guida:*
- Il modello 4-bit mostra “allucinazioni” (informazioni inventate) o omissioni?
- La coerenza e la qualità del riassunto peggiorano rispetto all’8-bit?

In [19]:
import ollama

# 1. Definizione del testo lungo (circa 250 parole)
testo_lungo = """
L'intelligenza artificiale generativa ha rivoluzionato il modo in cui interagiamo con i dati e con il codice.
Tuttavia, uno dei problemi principali legati all'adozione dei modelli linguistici di grandi dimensioni (LLM)
è il loro elevato requisito di risorse computazionali e di memoria. Per eseguire modelli con miliardi di parametri
su hardware locale o dispositivi edge, la ricerca si è concentrata su tecniche di compressione efficaci,
tra cui spicca la quantizzazione.

La quantizzazione riduce la precisione numerica dei pesi del modello, ad esempio convertendo i valori da punto
fluttuante a 16 bit (FP16) a numeri interi a 8 bit (INT8) o persino a 4 bit (INT4). Questo processo permette
di ridurre drasticamente l'occupazione della memoria RAM e VRAM, consentendo di eseguire modelli complessi su
dispositivi con risorse limitate e di velocizzare il tempo di generazione dei token.

Tuttavia, una compressione spinta a 4 bit comporta inevitabilmente dei rischi. Il degrado della precisione
può causare la perdita di sfumature semantiche, una minore capacità di ragionamento logico su contesti articolati
e un aumento della probabilità di allucinazioni, ovvero la generazione di informazioni errate o mai menzionate nel testo di partenza.
Valutare il bilanciamento tra l'efficienza dei tempi di inferenza e la fedeltà delle informazioni estratte è quindi un passo fondamentale
per definire l'applicabilità dei modelli quantizzati in scenari di produzione reali.
"""

# Prompt di stress per l'estrazione di punti chiave
long_prompt = f"Riassumi il seguente testo estraendo esattamente 5 punti chiave strutturati:\n\n{testo_lungo}"

# Modelli da confrontare
models = {"8bit": "llama3.2:3b-instruct-q8_0", "4bit": "llama3.2:3b-instruct-q4_0"}

risposte = {}

# 2. Esecuzione dell'inferenza sui due modelli
print("Esecuzione del test di stress in corso...\n")
for key, model_name in models.items():
  print(f"--- Generazione in corso con il modello {key} ({model_name}) ---")
  response = ollama.generate(model=model_name, prompt=long_prompt)
  risposte[key] = response["response"].strip()

# 3. Output comparativo
print("\n" + "=" * 80)
print("CONFRONTO RISPOSTE SU INPUT LUNGO (TEST DI STRESS)")
print("=" * 80)
print(f"\n[RISPOSTA MODELLO 8-BIT]:\n\n{risposte['8bit']}\n")
print("-" * 80)
print(f"\n[RISPOSTA MODELLO 4-BIT]:\n\n{risposte['4bit']}\n")
print("=" * 80)

Esecuzione del test di stress in corso...

--- Generazione in corso con il modello 8bit (llama3.2:3b-instruct-q8_0) ---
--- Generazione in corso con il modello 4bit (llama3.2:3b-instruct-q4_0) ---

CONFRONTO RISPOSTE SU INPUT LUNGO (TEST DI STRESS)

[RISPOSTA MODELLO 8-BIT]:

Ecco i 5 punti chiave estratti dal testo:

**1. La quantizzazione è una tecnica utilizzata per ridurre la memoria necessaria per eseguire modelli linguistici di grandi dimensioni (LLM)**

La quantizzazione riduce la precisione numerica dei pesi del modello, consentendo di ridurre l'occupazione della memoria RAM e VRAM.

**2. La quantizzazione può essere applicata su vari livelli di compressione, tra cui 16 bit (FP16), 8 bit (INT8) e 4 bit (INT4)**

La scelta del livello di compressione dipende dal bilanciamento tra efficienza dei tempi di inferenza e fedeltà delle informazioni estratte.

**3. La compressione spinta a 4 bit può comportare rischi, tra cui la perdita di sfumature semantiche e un aumento della probabi

## 🧩 Esercizio 5 – Misurare l’utilizzo di memoria in pratica

**Obiettivo:** verificare la riduzione del consumo RAM dovuta alla quantizzazione.

**Istruzioni:**
1. Avvia una sessione di inferenza con il **modello 8-bit** e poi con il **modello 4-bit**.
2. Durante l’esecuzione, osserva l’uso di memoria tramite strumenti di sistema (es. `top`, `htop` o Monitoraggio Attività su macOS).
3. Annota il valore massimo di RAM utilizzata e calcola la **differenza percentuale**.

💡 *Suggerimento tecnico:*
```python
import subprocess

def get_model_size_from_ps(model_name):
    """
    Legge l'output di 'ollama ps' e restituisce la memoria
    occupata dal modello (es. '2.8 GB').
    """

    result = subprocess.run(
        ["ollama", "ps"],
        capture_output=True,
        text=True
    )

    output = result.stdout

    for line in output.splitlines():

        if model_name in line:

            match = re.search(r"(\d+(\.\d+)?)\s*(GB|MB)", line)

            if match:
                return f"{match.group(1)} {match.group(3)}"

            return "Dimensione non trovata"

    return "Modello non trovato"

def unload_model(model_name):

    subprocess.run(
        ["ollama", "stop", model_name],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )


run_prompt(model, PROMPT)

time.sleep(2)

size = get_model_size_from_ps(model)

print(f"Memoria occupata: {size}")

results[model] = size

unload_model(model)  # ollama rimane in memoria per un po' il modello dopo averlo usato

time.sleep(5)

print(f"Memoria occupata: {size}")

```

📘 *Domande guida:*
- Quanta memoria risparmia il modello 4-bit?

In [20]:
import re
import subprocess
import time
import ollama

PROMPT = "Spiega brevemente che cos'è la quantizzazione dei modelli linguistici."

models = {
    "8bit": "llama3.2:3b-instruct-q8_0",
    "4bit": "llama3.2:3b-instruct-q4_0",
}


def get_model_size_from_ps(model_name):
  """Legge l'output di 'ollama ps' e restituisce la memoria occupata dal modello

  in GB (float) e la stringa formattata.
  """
  result = subprocess.run(
      ["ollama", "ps"], capture_output=True, text=True, check=False
  )
  output = result.stdout

  for line in output.splitlines():
    if model_name in line:
      match = re.search(r"(\d+(\.\d+)?)\s*(GB|MB)", line)
      if match:
        val = float(match.group(1))
        unit = match.group(3)
        size_gb = val if unit == "GB" else val / 1024.0
        return size_gb, f"{val} {unit}"

  return 0.0, "Modello non trovato"


def unload_model(model_name):
  """Rimuove il modello dalla memoria RAM/VRAM fermando il servizio di quell'istanza."""
  subprocess.run(
      ["ollama", "stop", model_name],
      stdout=subprocess.DEVNULL,
      stderr=subprocess.DEVNULL,
      check=False,
  )


memory_data = {}

print("--- INIZIO MISURAZIONE USO MEMORIA RAM ---")

for label, model_name in models.items():
  print(f"\n1. Avvio inferenza con modello {label} ({model_name})...")
  # Generazione per caricare il modello in memoria
  _ = ollama.generate(model=model_name, prompt=PROMPT)

  time.sleep(1)  # Breve pausa per stabilizzare l'allocazione

  # Lettura uso memoria
  size_gb, size_str = get_model_size_from_ps(model_name)
  memory_data[label] = size_gb

  print(f"   --> Memoria RAM/VRAM occupata da {label}: {size_str}")

  # Unload del modello per liberare la memoria prima del test successivo
  print(f"2. Scaricamento del modello {label} dalla memoria...")
  unload_model(model_name)
  time.sleep(3)

# --- TABELLA E CALCOLO RISPARMIO PERCENTUALE ---
ram_8bit = memory_data.get("8bit", 0.0)
ram_4bit = memory_data.get("4bit", 0.0)

print("\n" + "=" * 55)
print("RIEPILOGO CONSUMO DI MEMORIA")
print("=" * 55)
print(f"Modello 8-bit ({models['8bit']}): {ram_8bit:.2f} GB")
print(f"Modello 4-bit ({models['4bit']}): {ram_4bit:.2f} GB")

if ram_8bit > 0 and ram_4bit > 0:
  risparmio_assoluto = ram_8bit - ram_4bit
  risparmio_percentuale = (risparmio_assoluto / ram_8bit) * 100

  print("-" * 55)
  print(f"Risparmio di RAM in valore assoluto: {risparmio_assoluto:.2f} GB")
  print(f"Risparmio percentuale di memoria:   {risparmio_percentuale:.1f}%")
print("=" * 55)

--- INIZIO MISURAZIONE USO MEMORIA RAM ---

1. Avvio inferenza con modello 8bit (llama3.2:3b-instruct-q8_0)...
   --> Memoria RAM/VRAM occupata da 8bit: 4.4 GB
2. Scaricamento del modello 8bit dalla memoria...

1. Avvio inferenza con modello 4bit (llama3.2:3b-instruct-q4_0)...
   --> Memoria RAM/VRAM occupata da 4bit: 2.8 GB
2. Scaricamento del modello 4bit dalla memoria...

RIEPILOGO CONSUMO DI MEMORIA
Modello 8-bit (llama3.2:3b-instruct-q8_0): 4.40 GB
Modello 4-bit (llama3.2:3b-instruct-q4_0): 2.80 GB
-------------------------------------------------------
Risparmio di RAM in valore assoluto: 1.60 GB
Risparmio percentuale di memoria:   36.4%


# Laboratorio: Quantizzazione dei modelli LLM (4-bit vs 8-bit)

## 🧩 Esercizio 2 – Misurare la velocità di inferenza

**Obiettivo:** comprendere come la quantizzazione influisce sulla **velocità di risposta** del modello.

**Istruzioni:**
1. Usa lo **stesso prompt breve**, ad esempio:
   > "Chi ha scritto *I Promessi Sposi*?"
2. Misura il **tempo totale di generazione della risposta** per ciascuna variante: **modello 8-bit** e **modello 4-bit**.
3. Ripeti la misura **3 volte** per ogni modello e calcola la **media**.

💡 *Suggerimento tecnico:* puoi misurare il tempo con:
```python
import time

prompt = "Chi ha scritto I Promessi Sposi?"

# Misura per il modello 8-bit
t8 = []
for _ in range(3):
    start = time.time()
    _ = model_8bit(prompt)
    t8.append(time.time() - start)
print(f"Tempo medio 8-bit: {sum(t8)/3:.2f} s")

# Misura per il modello 4-bit
t4 = []
for _ in range(3):
    start = time.time()
    _ = model_4bit(prompt)
    t4.append(time.time() - start)
print(f"Tempo medio 4-bit: {sum(t4)/3:.2f} s")
```

📘 *Domande guida:*
- Quale versione è più veloce in media?
- Il guadagno di velocità corrisponde a  “riduzione dei tempi di inferenza”?


---

## 🧩 Esercizio 3 – Valutazione qualitativa della risposta (accuratezza)

**Obiettivo:** osservare l’effetto della quantizzazione sulla **qualità linguistica** delle risposte.

**Istruzioni:**
Prepara **5 prompt**, uno per ciascun tipo:
1. Domanda fattuale breve  
2. Spiegazione di un concetto semplice  
3. Riassunto di un testo di 6–8 frasi  
4. Produzione creativa (es. inizio di un racconto)  
5. Ragionamento leggero (es. “Se Marco ha 3 mele e ne dà una a Luca…”)

Esegui i prompt su **modello 8-bit** e **modello 4-bit**, confrontando:
- coerenza del testo,  
- completezza delle risposte,  
- correttezza delle informazioni.

💡 *Suggerimento tecnico:* salva le risposte in un dizionario per confrontarle facilmente:
```python
prompts = [
    "Chi ha inventato il telefono?",
    "Spiega in modo semplice cos'è una rete neurale.",
    "Riassumi questo testo: [incolla testo di 6-8 frasi]",
    "Scrivi l'inizio di una storia ambientata su Marte.",
    "Se Marco ha 3 mele e ne dà una a Luca, quante ne rimangono?"
]

risposte = {"8bit": [], "4bit": []}
for p in prompts:
    risposte["8bit"].append(model_8bit(p))
    risposte["4bit"].append(model_4bit(p))
```

📘 *Domande guida:*
- Quale versione produce risposte più coerenti e accurate?
- Si nota perdita di informazione o ripetizioni nella versione 4-bit?
- Come si manifesta il “trade-off” tra efficienza e qualità?

---

## 🧩 Esercizio 4 – Prompt di stress (robustezza con input lunghi)

**Obiettivo:** valutare come la **quantizzazione più spinta (4-bit)** reagisce a input complessi.

**Istruzioni:**
1. Prendi un testo di circa **200–300 parole** (es. un breve articolo o un estratto da Wikipedia).
2. Crea un prompt che chieda di **riassumere** o **estrarre i punti chiave**.
3. Esegui lo stesso prompt con **modello 8-bit** e **modello 4-bit**.

💡 *Suggerimento tecnico:*
```python
long_prompt = "Riassumi il seguente testo in 5 punti chiave:\n\n[TUO TESTO QUI]"
print("Risposta 8-bit:")
print(model_8bit(long_prompt))
print("\nRisposta 4-bit:")
print(model_4bit(long_prompt))
```

📘 *Domande guida:*
- Il modello 4-bit mostra “allucinazioni” (informazioni inventate) o omissioni?
- La coerenza e la qualità del riassunto peggiorano rispetto all’8-bit?

---

## 🧩 Esercizio 5 – Misurare l’utilizzo di memoria in pratica

**Obiettivo:** verificare la riduzione del consumo RAM dovuta alla quantizzazione.

**Istruzioni:**
1. Avvia una sessione di inferenza con il **modello 8-bit** e poi con il **modello 4-bit**.
2. Durante l’esecuzione, osserva l’uso di memoria tramite strumenti di sistema (es. `top`, `htop` o Monitoraggio Attività su macOS).
3. Annota il valore massimo di RAM utilizzata e calcola la **differenza percentuale**.

💡 *Suggerimento tecnico:*
```python
import subprocess

def get_model_size_from_ps(model_name):
    """
    Legge l'output di 'ollama ps' e restituisce la memoria
    occupata dal modello (es. '2.8 GB').
    """

    result = subprocess.run(
        ["ollama", "ps"],
        capture_output=True,
        text=True
    )

    output = result.stdout

    for line in output.splitlines():

        if model_name in line:

            match = re.search(r"(\d+(\.\d+)?)\s*(GB|MB)", line)

            if match:
                return f"{match.group(1)} {match.group(3)}"

            return "Dimensione non trovata"

    return "Modello non trovato"

def unload_model(model_name):

    subprocess.run(
        ["ollama", "stop", model_name],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )


run_prompt(model, PROMPT)

time.sleep(2)

size = get_model_size_from_ps(model)

print(f"Memoria occupata: {size}")

results[model] = size

unload_model(model)  # ollama rimane in memoria per un po' il modello dopo averlo usato

time.sleep(5)

print(f"Memoria occupata: {size}")

```

📘 *Domande guida:*
- Quanta memoria risparmia il modello 4-bit?

---

## 🧩 Esercizio 6 – Stabilità su sessioni prolungate

**Obiettivo:** valutare la **coerenza nel tempo** dei modelli quantizzati.

**Istruzioni:**
1. Scegli un argomento (es. “Rivoluzione industriale” o “Evoluzione delle stelle”).
2. Simula una **conversazione di 10–15 turni** con entrambi i modelli.
3. Analizza se il modello 4-bit perde coerenza o produce risposte più vaghe col passare dei turni.

💡 *Suggerimento tecnico:* puoi simulare una conversazione concatenando le risposte:
```python
DOMANDE = [
    "Parliamo dell'evoluzione delle stelle. Dammi una panoramica generale.",
    "Come nasce una stella?",
    "Cos'è una nebulosa?",
    "Qual è il ruolo della gravità?",
    "Che differenza c'è tra una protostella e una stella adulta?",
    "Come funziona la fusione nucleare?"
    "Come si forma un buco nero stellare?",
    "Riassumi tutto quello che abbiamo discusso finora.",
    "Puoi costruire una timeline delle fasi evolutive?",
]

def run_conversation(model_name):

    messages = []

    transcript = []

    for turn, question in enumerate(DOMANDE, start=1):

        messages.append(
            {
                "role": "user",
                "content": question
            }
        )

        response = ollama.chat(
            model=model_name,
            messages=messages
        )

        answer = response["message"]["content"]

        messages.append(
            {
                "role": "assistant",
                "content": answer
            }
        )

        transcript.append({
            "turn": turn,
            "question": question,
            "answer": answer
        })

        print(f"\n=== TURNO {turn} ===")
        # print(f"DOMANDA: {question}")
        # print(f"RISPOSTA:\n{answer}")

    return transcript
```

📘 *Domande guida:*
- Il modello 4-bit rimane coerente sul tema?
- Ripete concetti o cambia argomento più spesso rispetto all’8-bit?
- Collega le osservazioni alle *best practices* di test dei modelli quantizzati.

---

## 🧩 Esercizio 7 – Valutazione della latenza percepita

**Obiettivo:** misurare la **reattività percepita** dal punto di vista dell’utente.

**Istruzioni:**
1. Cronometra:
   - il tempo tra l’invio del prompt e la comparsa del **primo token**,
   - il tempo totale per una risposta di circa **150 parole**.
2. Ripeti per **modello 8-bit** e **modello 4-bit**.
3. Valuta la “usabilità” con la scala:
   - 🔵 Ottimo: avvio < 1 s  
   - 🟢 Accettabile: 1–3 s  
   - 🔴 Scarso: > 3 s

💡 *Suggerimento tecnico:*
```python
start = time.time()
_ = model_8bit("Spiega in 150 parole come funziona la quantizzazione nei LLM.")
t_8 = time.time() - start

start = time.time()
_ = model_4bit("Spiega in 150 parole come funziona la quantizzazione nei LLM.")
t_4 = time.time() - start

print(f"Tempo totale 8-bit: {t_8:.2f}s")
print(f"Tempo totale 4-bit: {t_4:.2f}s")
```

📘 *Domande guida:*
- Quale modello risponde più rapidamente?
- La differenza di latenza è percepibile per un utente umano?
- In che misura la riduzione di precisione migliora la **usabilità pratica**?
